# Day 25: Action Recognition using YOLO11 Pose

**Objective:** Convert human pose keypoints into action prediction (Standing, Walking, Running, etc.)

**Key Idea:**  
Pose keypoints over time → Sequence → Action Classifier

In [ ]:
!pip install ultralytics pandas scikit-learn -q
print("✅ Installed")

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
from collections import deque
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

print("✅ Libraries imported")

In [ ]:
model = YOLO("yolo11s-pose.pt")
print("✅ YOLO11 Pose model loaded")

In [ ]:
def extract_pose_sequence(video_path, max_frames=150):
    """Extract keypoints sequence from video"""
    cap = cv2.VideoCapture(video_path)
    pose_sequence = []
    frame_count = 0
    
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
            
        results = model(frame, conf=0.5, verbose=False)
        
        if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
            # Take the first detected person
            keypoints = results[0].keypoints.data[0].cpu().numpy()  # (17, 3) -> x, y, confidence
            pose_sequence.append(keypoints.flatten())  # Flatten to 51 features
        
        frame_count += 1
    
    cap.release()
    return np.array(pose_sequence)

print("✅ Pose extraction function ready")

In [ ]:
# For now, we create a dummy classifier (you can train real one later)
# In real project, you collect many labeled videos

def predict_action(pose_sequence):
    """Very simple rule-based + ML action predictor"""
    if len(pose_sequence) < 10:
        return "Unknown"
    
    # Simple heuristic: Check vertical movement of ankles/hips
    ankle_y_movement = np.std(pose_sequence[:, 15*3+1::3])  # y-coordinate of ankles
    
    if ankle_y_movement < 5:
        return "Standing"
    elif ankle_y_movement < 25:
        return "Walking"
    else:
        return "Running / Fast Movement"

# Test
action = predict_action(sequence)
print(f"Predicted Action: **{action}**")